To use Microsoft Agent Framework with Azure OpenAI, you need to install the following Python packages:

In [ ]:
%pip install agent-framework==1.0.0b251223

Note: you may need to restart the kernel to use updated packages.


## Create the agent

First, create a chat client for communicating with Azure OpenAI and use the same login as you used when authenticating with the Azure CLI in the Prerequisites step.
Then, create the agent, providing instructions and a name for the agent.

In [2]:
import os

os.environ["NO_PROXY"] = "*"

In [ ]:
%pip install agent-framework==1.0.0b251216
# %pip install agent-framework --pre

In [3]:
from dotenv import load_dotenv
import os

if os.path.exists(".env"):
    load_dotenv(override=True)

In [4]:
from agent_framework.azure import AzureOpenAIChatClient

agent = AzureOpenAIChatClient(endpoint=os.environ["AZURE_OPENAI_ENDPOINT"], 
                              deployment_name="gpt-4o-mini", 
                              api_key=os.environ["AZURE_OPENAI_API_KEY"]
                             ).create_agent(
    instructions="You are good at telling jokes.",
    name="Joker"
)

result = await agent.run("Tell me a joke about a pirate.")
print(result.text)

Why did the pirate go to school?

Because he wanted to improve his "arrrrrticulation"!


This demo shows you how to use images with an agent, allowing the agent to analyze and respond to image content.

In [5]:
agent = AzureOpenAIChatClient(endpoint=os.environ["AZURE_OPENAI_ENDPOINT"], 
                              deployment_name="gpt-4o-mini", 
                              api_key=os.environ["AZURE_OPENAI_API_KEY"]).create_agent(
    name="VisionAgent",
    instructions="You are a helpful agent that can analyze images"
)

from agent_framework import ChatMessage, TextContent, UriContent, Role

message = ChatMessage(
    role=Role.USER,
    contents=[
        TextContent(text="What do you see in this image?"),
        UriContent(
            uri="https://upload.wikimedia.org/wikipedia/commons/thumb/d/dd/Gfp-wisconsin-madison-the-nature-boardwalk.jpg/2560px-Gfp-wisconsin-madison-the-nature-boardwalk.jpg",
            media_type="image/jpeg"
        )
    ]
)

result = await agent.run(message)
print(result.text)

ServiceResponseException: <class 'agent_framework.azure._chat_client.AzureOpenAIChatClient'> service failed to complete the prompt: Error code: 400 - {'error': {'code': 'BadRequest', 'message': 'The provided image url can not be accessed. status code: 429.', 'param': None, 'type': None}}

## Running the agent with a multi-turn conversation

Agents are stateless and do not maintain any state internally between calls. To have a multi-turn conversation with an agent, you need to create an object to hold the conversation state and pass this object to the agent when running it.

To create the conversation state object, call the GetNewThread method on the agent instance.

In [8]:
thread = agent.get_new_thread()

You can then pass this thread object to the run and run_stream methods on the agent instance, along with the user input.

In [11]:

result1 = await agent.run("Tell me a joke about a pirate.", thread=thread)
print(result1.text)

result2 = await agent.run("Now add some emojis to the joke and tell it in the voice of a pirate's parrot.", thread=thread)
print(result2.text)

Why did the pirate go to school?

Because he wanted to improve his "arrrticulation"!
Squawk! 🦜 Why did the pirate go to school? 📚

Because he wanted to improve his "arrrticulation"! 😂 Arrr!


List the messages stored i the `thread` object to see the full conversation history.

In [ ]:
for message in await thread.message_store.list_messages():
    print(f"{message.role}: {message.contents[0].text}")

user: Tell me a joke about a pirate.
assistant: Why did the pirate go to school?

Because he wanted to improve his "arrrticulation"!
user: Now add some emojis to the joke and tell it in the voice of a pirate's parrot.
assistant: Squawk! 🦜 Why did the pirate go to school? 📚

Because he wanted to improve his "arrrticulation"! 😂 Arrr!


## Single agent with multiple conversations

It is possible to have multiple, independent conversations with the same agent instance, by creating multiple AgentThread objects. These threads can then be used to maintain separate conversation states for each conversation. The conversations will be fully independent of each other, since the agent does not maintain any state internally.

In [21]:
thread1 = agent.get_new_thread()
thread2 = agent.get_new_thread()

result1 = await agent.run("Tell me a joke about a pirate.", thread=thread1)
print(result1.text)

result2 = await agent.run("Tell me a joke about a robot.", thread=thread2)
print(result2.text)

result3 = await agent.run("Now add some emojis to the joke and tell it in the voice of a pirate's parrot.", thread=thread1)
print(result3.text)

result4 = await agent.run("Now add some emojis to the joke and tell it in the voice of a robot.", thread=thread2)
print(result4.text)

Why did the pirate go to school?

Because he wanted to improve his "arrrticulation"!
Why did the robot go on a diet?

Because it had too many bytes!
🦜 Arrr, matey! Here be a joke fer ye! 

Why did the pirate go to school? 🤔🏴‍☠️

Because he wanted to improve his "arrrticulation"! 😂📚✨
Beep boop! 🤖 Why did the robot go on a diet? 🤔 Because it had too many bytes! 🍔💻 Beep!


## Using function tools with an agent

This tutorial step shows you how to use function tools with an agent, where the agent is built on the Azure OpenAI Chat Completion service.

Function tools are just custom code that you want the agent to be able to call when needed. You can turn any Python function into a function tool by passing it to the agent's tools parameter when creating the agent.

If you need to provide additional descriptions about the function or its parameters to the agent, so that it can more accurately choose between different functions, you can use Python's type annotations with Annotated and Pydantic's Field to provide descriptions.

Here is an example of a simple function tool that fakes getting the weather for a given location. It uses type annotations to provide additional descriptions about the function and its location parameter to the agent.

In [22]:
from typing import Annotated
from pydantic import Field

def get_weather(
    location: Annotated[str, Field(description="The location to get the weather for.")],
) -> str:
    """Get the weather for a given location."""
    return f"The weather in {location} is cloudy with a high of 15°C."

You can also use the ai_function decorator to explicitly specify the function's name and description:

In [23]:
from typing import Annotated
from pydantic import Field
from agent_framework import ai_function

@ai_function(name="weather_tool", description="Retrieves weather information for any location")
def get_weather(
    location: Annotated[str, Field(description="The location to get the weather for.")],
) -> str:
    return f"The weather in {location} is cloudy with a high of 15°C."

If you don't specify the name and description parameters in the ai_function decorator, the framework will automatically use the function's name and docstring as fallbacks.

When creating the agent, you can now provide the function tool to the agent, by passing it to the tools parameter.

In [25]:
from agent_framework.azure import AzureOpenAIChatClient
from azure.identity import AzureCliCredential

agent = AzureOpenAIChatClient(endpoint=os.environ["AZURE_OPENAI_ENDPOINT"], 
                              deployment_name="gpt-4o-mini", 
                              api_key=os.environ["AZURE_OPENAI_API_KEY"]).create_agent(
    instructions="You are a helpful assistant",
    tools=get_weather
)

Now you can just run the agent as normal, and the agent will be able to call the get_weather function tool when needed.

In [26]:
result = await agent.run("What is the weather like in Amsterdam?")
print(result.text)

The weather in Amsterdam is currently cloudy, with a high temperature of 15°C.


## Create a class with multiple function tools

You can also create a class that contains multiple function tools as methods. This can be useful for organizing related functions together or when you want to pass state between them.

In [47]:
class WeatherTools:
    def __init__(self):
        self.last_location = None

    @ai_function(name="weather_tool", description="Retrieves weather information for any location")
    def get_weather(
        self,
        location: Annotated[str, Field(description="The location to get the weather for.")],
    ) -> str:
        """Get the weather for a given location."""
        return f"The weather in {location} is cloudy with a high of 15°C."

    @ai_function(name="weather_tool_details", description="Get the detailed weather for the last requested location.")
    def get_weather_details(self) -> int:
        """Get the detailed weather for the last requested location."""
        if self.last_location is None:
            return "No location specified yet."
        return f"The detailed weather in {self.last_location} is cloudy with a high of 15°C, low of 7°C, and 60% humidity."

When creating the agent, you can now provide all the methods of the class as functions:

In [48]:
tools = WeatherTools()
agent = AzureOpenAIChatClient(endpoint=os.environ["AZURE_OPENAI_ENDPOINT"], 
                              deployment_name="gpt-4o-mini", 
                              api_key=os.environ["AZURE_OPENAI_API_KEY"]).create_agent(
    instructions="You are a helpful assistant",
    tools=[tools.get_weather, tools.get_weather_details]
)

In [61]:
thread = agent.get_new_thread()

result1 = await agent.run("What is the weather like in Amsterdam?", thread=thread)
print(result1.text)

result2 = await agent.run("What is the detailed weather like in Amsterdam?", thread=thread)
print(result2.text)


The weather in Amsterdam is currently cloudy, with a high temperature of 15°C.



## 5. Using function tools with human in the loop approvals

This tutorial step shows you how to use function tools that require human approval with an agent.

When agents require any user input, for example to approve a function call, this is referred to as a human-in-the-loop pattern. An agent run that requires user input, will complete with a response that indicates what input is required from the user, instead of completing with a final answer. The caller of the agent is then responsible for getting the required input from the user, and passing it back to the agent as part of a new agent run.

### Create the agent with function tools requiring approval

When using functions, it's possible to indicate for each function, whether it requires human approval before being executed. This is done by setting the approval_mode parameter to "always_require" when using the @ai_function decorator.

Here is an example of a simple function tool that fakes getting the weather for a given location.

In [50]:
from typing import Annotated
from agent_framework import ai_function

@ai_function
def get_weather(location: Annotated[str, "The city and state, e.g. San Francisco, CA"]) -> str:
    """Get the current weather for a given location."""
    return f"The weather in {location} is cloudy with a high of 15°C."

To create a function that requires approval, you can use the approval_mode parameter:



In [51]:
@ai_function(approval_mode="always_require")
def get_weather_detail(location: Annotated[str, "The city and state, e.g. San Francisco, CA"]) -> str:
    """Get detailed weather information for a given location."""
    return f"The weather in {location} is cloudy with a high of 15°C, humidity 88%."

When creating the agent, you can now provide the approval requiring function tool to the agent, by passing a list of tools to the ChatAgent constructor.
Since you now have a function that requires approval, the agent might respond with a request for approval, instead of executing the function directly and returning the result. You can check the response for any user input requests, which indicates that the agent requires user approval for a function.

In [ ]:
from agent_framework import ChatAgent
from agent_framework.openai import OpenAIResponsesClient

agent = AzureOpenAIChatClient(endpoint=os.environ["AZURE_OPENAI_ENDPOINT"], 
                              deployment_name="gpt-4o-mini", 
                              api_key=os.environ["AZURE_OPENAI_API_KEY"]).create_agent(
    instructions="You are a helpful weather assistant.",
    tools=[get_weather, get_weather_detail]
)
 
result = await agent.run("What is the detailed weather like in Amsterdam?")

if result.user_input_requests:
    for user_input_needed in result.user_input_requests:
        print(f"Function: {user_input_needed.function_call.name}")
        print(f"Arguments: {user_input_needed.function_call.arguments}")

Function: get_weather_detail
Arguments: {"location":"Amsterdam"}


If there are any function approval requests, the detail of the function call including name and arguments can be found in the function_call property on the user input request. This can be shown to the user, so that they can decide whether to approve or reject the function call.

Once the user has provided their input, you can create a response using the create_response method on the user input request. Pass True to approve the function call, or False to reject it.

The response can then be passed to the agent in a new ChatMessage, to get the result back from the agent.

In [58]:
from agent_framework import ChatMessage, Role

# Get user approval (in a real application, this would be interactive)
user_approval = True  # or False to reject

# Create the approval response
approval_message = ChatMessage(
    role=Role.USER, 
    contents=[user_input_needed.create_response(user_approval)]
)

# Continue the conversation with the approval
final_result = await agent.run([
    "What is the detailed weather like in Amsterdam?",
    ChatMessage(role=Role.ASSISTANT, contents=[user_input_needed]),
    approval_message
])
print(final_result.text)

The detailed weather in Amsterdam is currently cloudy, with a high temperature of 15°C and a humidity level of 88%.


## 6. Create the agent with structured output

The ChatAgent is built on top of any chat client implementation that supports structured output. The ChatAgent uses the response_format parameter to specify the desired output schema.

When creating or running the agent, you can provide a Pydantic model that defines the structure of the expected output.

Various response formats are supported based on the underlying chat client capabilities.

This example creates an agent that produces structured output in the form of a JSON object that conforms to a Pydantic model schema.

First, define a Pydantic model that represents the structure of the output you want from the agent:

In [4]:
from pydantic import BaseModel

class PersonInfo(BaseModel):
    """Information about a person."""
    name: str | None = None
    age: int | None = None
    occupation: str | None = None

Now you can create an agent using the Azure OpenAI Chat Client:

In [5]:
from agent_framework.azure import AzureOpenAIChatClient
from azure.identity import AzureCliCredential

# Create the agent using Azure OpenAI Chat Client
agent = AzureOpenAIChatClient(endpoint=os.environ["AZURE_OPENAI_ENDPOINT"], 
                              deployment_name="gpt-4o-mini", 
                              api_key=os.environ["AZURE_OPENAI_API_KEY"]).create_agent(
    name="HelpfulAssistant",
    instructions="You are a helpful assistant that extracts person information from text."
)

Now you can run the agent with some textual information and specify the structured output format using the response_format parameter:

In [6]:
response = await agent.run(
    "Please provide information about John Smith, who is a 35-year-old software engineer.",
    response_format=PersonInfo
)

The agent response will contain the structured output in the value property, which can be accessed directly as a Pydantic model instance:

In [7]:
if response.value:
    person_info = response.value
    print(f"Name: {person_info.name}, Age: {person_info.age}, Occupation: {person_info.occupation}")
else:
    print("No structured data found in response")

Name: John Smith, Age: 35, Occupation: software engineer


## 7. Using an agent as a function tool

This tutorial shows you how to use an agent as a function tool, so that one agent can call another agent as a tool.

You can use a ChatAgent as a function tool by calling .as_tool() on the agent and providing it as a tool to another agent. This allows you to compose agents and build more advanced workflows.

First, create a function tool that will be used by your agent that's exposed as a function.

In [8]:
from typing import Annotated
from pydantic import Field

def get_weather(
    location: Annotated[str, Field(description="The location to get the weather for.")],
) -> str:
    """Get the weather for a given location."""
    return f"The weather in {location} is cloudy with a high of 15°C."

Create a ChatAgent that uses the function tool.

In [9]:
from agent_framework.azure import AzureOpenAIChatClient
from azure.identity import AzureCliCredential

weather_agent = AzureOpenAIChatClient(
    endpoint=os.environ["AZURE_OPENAI_ENDPOINT"], 
    deployment_name="gpt-4o-mini", 
    api_key=os.environ["AZURE_OPENAI_API_KEY"]
).create_agent(
    name="WeatherAgent",
    description="An agent that answers questions about the weather.",
    instructions="You answer questions about the weather.",
    tools=get_weather
)

Now, create a main agent and provide the weather_agent as a function tool by calling .as_tool() to convert weather_agent to a function tool.

In [10]:
main_agent = AzureOpenAIChatClient(
    endpoint=os.environ["AZURE_OPENAI_ENDPOINT"], 
    deployment_name="gpt-4o-mini", 
    api_key=os.environ["AZURE_OPENAI_API_KEY"]
).create_agent(
    instructions="You are a helpful assistant who responds in French.",
    tools=weather_agent.as_tool()
)

Invoke the main agent as normal. It can now call the weather agent as a tool, and should respond in French.

In [11]:
result = await main_agent.run("What is the weather like in Amsterdam?")
print(result.text)

Le temps actuel à Amsterdam est nuageux, avec une température maximale de 15 °C.


## 8. Expose an agent as an MCP tool

This tutorial shows you how to expose an agent as a tool over the Model Context Protocol (MCP), so it can be used by other systems that support MCP tools.

### Expose an agent as an MCP server

You can expose an agent as an MCP server by using the as_mcp_server() method. This allows the agent to be invoked as a tool by any MCP-compatible client.

First, create an agent that you'll expose as an MCP server. You can also add tools to the agent:

In [27]:
from typing import Annotated
from agent_framework.openai import OpenAIResponsesClient

def get_specials() -> Annotated[str, "Returns the specials from the menu."]:
    return """
        Special Soup: Clam Chowder
        Special Salad: Cobb Salad
        Special Drink: Chai Tea
        """

def get_item_price(
    menu_item: Annotated[str, "The name of the menu item."],
) -> Annotated[str, "Returns the price of the menu item."]:
    return "$9.99"

# Create an agent with tools
agent = AzureOpenAIChatClient(
# agent = OpenAIResponsesClient(
    endpoint=os.environ["AZURE_OPENAI_ENDPOINT"], 
    deployment_name="gpt-4o-mini", 
    api_key=os.environ["AZURE_OPENAI_API_KEY"]
).create_agent(
    name="RestaurantAgent",
    description="Answer questions about the menu.",
    tools=[get_specials, get_item_price],
)

Turn the agent into an MCP server. The agent name and description will be used as the MCP server metadata:

In [28]:
# Expose the agent as an MCP server
server = agent.as_mcp_server()

Setup the MCP server to listen for incoming requests over standard input/output:

In [35]:
import anyio
import asyncio
from mcp.server.stdio import stdio_server

async def run():
    async def handle_stdin():
        async with stdio_server() as (read_stream, write_stream):
            await server.run(read_stream, write_stream, server.create_initialization_options())

    await handle_stdin()

if __name__ == "__main__":
    await run()
    # anyio.run(run)

ValueError: I/O operation on closed file

In [39]:
! python agent_as_mcp_server.py

^C


Now your agent listens for MCP JSON-RPC requests over stdio.
Clients can call it using MCP-compliant tools (VS Code, MCP Inspector, or your own client code).

In [ ]:
{
	"servers": {
		"my-mcp-server": {
			"type": "stdio",
			"command": "python",
			"args": ["D:/projects/ai-course/200_agentic_framework_hello/agent_as_mcp_server.py"],
			"dev": {
				"watch": "**/*.py",
				"debug": {
				"type": "debugpy",
        }
      }
		}
	},
	"inputs": []
}																										

In [ ]:
! code --add-mcp "{\"name\":\"my-server\",\"command\": \"uvx\",\"args\": [\"mcp-server-fetch\"]}"

## 9. Enabling observability for Agents

This tutorial shows how to enable OpenTelemetry on an agent so that interactions with the agent are automatically logged and exported. In this tutorial, output is written to the console using the OpenTelemetry console exporter.

### Enable OpenTelemetry in your app

The simplest way to enable observability is using `configure_otel_providers()`.
Two samples are available in the python files `agent_observability_otel_console.py` and `agent_observability_otel_appinsights.py`.
The first sample shows how to configure OpenTelemetry to use the console exporter, while the second sample shows how to configure OpenTelemetry to use the Azure Application Insights exporter.

### 9.1. Running the sample with console exporter

Let's try to run the console exporter sample.
Make sure first to install the dependencies.

In [ ]:
%pip install opentelemetry-sdk

In [27]:
! python agent_observability_otel_console.py

d:\projects\ai-course\200_agentic_framework_hello\agent_observability_otel_console.py:35: DeprecationWarning: Use ConsoleLogRecordExporter. Since logs are not stable yet this WILL be removed in future releases.
  exporter = ConsoleLogExporter()
Traceback (most recent call last):
  File "c:\Users\hodellai\AppData\Local\Programs\Python\Python313\Lib\site-packages\agent_framework\observability.py", line 327, in _create_otlp_exporters
    from opentelemetry.exporter.otlp.proto.grpc._log_exporter import OTLPLogExporter as GRPCLogExporter
ModuleNotFoundError: No module named 'opentelemetry.exporter'

The above exception was the direct cause of the following exception:

Traceback (most recent call last):
  File "d:\projects\ai-course\200_agentic_framework_hello\agent_observability_otel_console.py", line 106, in <module>
    configure_otel_providers(
    ~~~~~~~~~~~~~~~~~~~~~~~~^
        enable_sensitive_data=True,   # development only
        ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
 

The console exporter will show trace data similar to:

```json
{
    "name": "chat gpt-4o-mini",
    "context": {
        "trace_id": "0xee18eb462f337d4acc8fa57b86dea6f9",
        "span_id": "0x225963c0c41548af",
        "trace_state": "[]"
    },
    "kind": "SpanKind.INTERNAL",
    "parent_id": "0x044791c43b8b5656",
    "start_time": "2025-12-27T12:01:41.594873Z",
    "end_time": "2025-12-27T12:01:43.102008Z",
    "status": {
        "status_code": "UNSET"
    },
    "attributes": {
        "gen_ai.input.messages": "[{\"role\": \"user\", \"parts\": [{\"type\": \"text\", \"content\": \"Tell me a joke about a pirate.\"}]}]",
        "gen_ai.operation.name": "chat",
        "gen_ai.request.choice.count": 1,
        "gen_ai.provider.name": "azure.ai.openai",
        "gen_ai.request.model": "gpt-4o-mini",
        "server.address": "https://ai-services-333-400.openai.azure.com/openai/deployments/gpt-4o-mini/",
        "gen_ai.response.id": "chatcmpl-CrNHK1I9EVf0H3hYLApDLQz9oCW1K",
        "gen_ai.response.finish_reasons": "[\"stop\"]",
        "gen_ai.response.model": "gpt-4o-mini-2024-07-18",
        "gen_ai.usage.input_tokens": 26,
        "gen_ai.usage.output_tokens": 22,
        "gen_ai.client.operation.duration": 1.5061967000365257,
        "gen_ai.output.messages": "[{\"role\": \"assistant\", \"parts\": [{\"type\": \"text\", \"content\": \"Why did the pirate go to school? \\n\\nBecause he wanted to improve his \\\"arrrticulation!\\\"\"}], \"finish_reason\": \"stop\"}]"
    },
    "events": [],
    "links": [],
    "resource": {
        "attributes": {
            "telemetry.sdk.language": "python",
            "telemetry.sdk.name": "opentelemetry",
            "telemetry.sdk.version": "1.39.1",
            "service.name": "telemetry-agent-framework"
        },
        "schema_url": ""
    }
}
```

### 9.2. Running the sample with Azure Application Insights exporter

To run the sample with Azure Application Insights exporter, make sure you have the `azure-monitor-opentelemetry` package installed. And also make sure you have an Application Insights resource created in your Azure subscription. Application Insights requires an instrumentation key or Connection String to send telemetry data to the correct resource. You can find the Connection String in the "Overview" section of your Application Insights resource in the Azure portal. Copy the Connection String value.

In [25]:
%pip install azure-monitor-opentelemetry==1.8.3

  Using cached msrest-0.7.1-py3-none-any.whl.metadata (21 kB)
Using cached msrest-0.7.1-py3-none-any.whl (85 kB)

   ----------------------------------------  0/22 [wrapt]
   - --------------------------------------  1/22 [opentelemetry-util-http]
   --- ------------------------------------  2/22 [asgiref]
   --- ------------------------------------  2/22 [asgiref]
   --- ------------------------------------  2/22 [asgiref]
  Attempting uninstall: opentelemetry-api
   --- ------------------------------------  2/22 [asgiref]
    Found existing installation: opentelemetry-api 1.39.1
   --- ------------------------------------  2/22 [asgiref]
    Uninstalling opentelemetry-api-1.39.1:
   --- ------------------------------------  2/22 [asgiref]
   ----- ----------------------------------  3/22 [opentelemetry-api]
      Successfully uninstalled opentelemetry-api-1.39.1
   ----- ----------------------------------  3/22 [opentelemetry-api]
   ----- ----------------------------------  3/22 [op

Now you can run the Application Insights exporter sample. It would be better if you run it from within a Terminal.

In [29]:
! python agent_observability_otel_appinsights.py

Traceback (most recent call last):
  File "c:\Users\hodellai\AppData\Local\Programs\Python\Python313\Lib\site-packages\agent_framework\observability.py", line 327, in _create_otlp_exporters
    from opentelemetry.exporter.otlp.proto.grpc._log_exporter import OTLPLogExporter as GRPCLogExporter
ModuleNotFoundError: No module named 'opentelemetry.exporter'

The above exception was the direct cause of the following exception:

Traceback (most recent call last):
  File "d:\projects\ai-course\200_agentic_framework_hello\agent_observability_otel_appinsights.py", line 35, in <module>
    configure_otel_providers(
    ~~~~~~~~~~~~~~~~~~~~~~~~^
            enable_sensitive_data=True, # development only
            ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
            vs_code_extension_port=4317,  # Connects to AI Toolkit
            ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
        )
        ^
  File "c:\Users\hodellai\AppData\Local\Programs\Python\Python313\Lib\site-packages\a

You can now view the metrics collected in the Azure portal under your Application Insights resource. It may take a few minutes for the data to appear.

![Application Insights Metrics](./images/observability-app-insights.png)

![Grafana Metrics](./images/observability-grafana.png)